<center><font size=20> <bold> AI Business Research Agent </font size> </bold></center>


## **Objective**



Build an AI-powered research agent that can find, verify, organize, and
summarize business information from publicly available internet sources.
The agent should behave like a professional researcher rather than a simple web
scraper.


## **Installing Packages**

The necessary packages are installed.

*  google-generativeai -- lets talk to gemini model
*  ddgs -- access to DuckDuckGo search
*  pandas -- for business records into tables
*  requests -- lets python to visit webpage and download raw contents
*  beautifulsoup4 - convert raw html to readable
*  rapidfuzz - for dedeuplication

In [1]:
!pip install -q google-generativeai
!pip install -q ddgs
!pip install -q pandas
!pip install -q requests
!pip install -q beautifulsoup4
!pip install -q rapidfuzz

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 70.6/70.6 kB 2.7 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 161.7/161.7 kB 6.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 5.3/5.3 MB 58.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 29.1 MB/s eta 0:00:00


In [2]:
import pandas as pd
import requests
import re
from urllib.parse import urlparse
from bs4 import BeautifulSoup

## **Gemini Setup**

Gemini model has been connected succesfully which is used at the end of the pipeline to generate a professional research summary report from the collected business data.

In [3]:
from google.colab import userdata
import google.generativeai as genai
api_key = userdata.get("GEMINI_API_KEY")
genai.configure(api_key=api_key)
model = genai.GenerativeModel("gemini-2.5-flash")
print("Gemini conneced successfully!")

/usr/local/lib/python3.12/dist-packages/google/colab/_import_hooks/_hook_injector.py:55: FutureWarning: 

All support for the `google.generativeai` package has ended. It will no longer be receiving 
updates or bug fixes. Please switch to the `google.genai` package as soon as possible.
See README for more details:

https://github.com/google-gemini/deprecated-generative-ai-python/blob/main/README.md

  loader.exec_module(module)


Gemini conneced successfully!


## **Utility Functions**

### **Helper Functions 1: Extract Domain from URL**

Extracts the domain name from a given URL. This standardizes website references and enables consistent source classification and business identification across different web pages.

In [4]:
from urllib.parse import urlparse
def extract_domain(url):
  try:
    domain = urlparse(url).netloc
    domain = domain.replace("www.", "")
    return domain
  except:
    return ""

In [5]:
print(extract_domain("https://www.healthpartners.com"))

healthpartners.com


### **Helper Functions 2 : Domain to Business Name**

Converts a website domain into a human readable business name. This help generate business records when company names are not explicitly available from the search results.

In [6]:
def domain_to_business_name(domain):
  parts = domain.split(".")
  if len(parts) >=2:
    name = parts[-2]
  else:
    name = parts[0]
  return(name.replace("-"," ").title())

In [7]:
print(domain_to_business_name("mayoclinic.org"))
print(domain_to_business_name("healthpartners.com"))
print(domain_to_business_name("providers.mhealthfairview.org"))

Mayoclinic
Healthpartners
Mhealthfairview


### **Helper Functions 3 : Web Page Text Collection**

Retrieves and extracts visible text contect from a webpage. The collected text is later used for contact information, business verification, and information analysis.

In [8]:
def collect_page_text(url):
  try:
    response = requests.get(url, headers = {"User-Agent": "Mozilla/5.0"}, timeout=10)
    if response.status_code !=200:
      return None
    soup = BeautifulSoup(response.text, "html.parser")
    text = soup.get_text(separator=" ", strip=True)
    return text[:10000]
  except Exception:
    return None

### **Helper Function 4 : Email Extraction**

Uses regular expressions to scan webpage text and identify email addresses. Handles both standard formats like .com and international domains like .in, .co.uk. Duplicate emails are automatically removed.

In [9]:
def extract_emails(text):
    if not text:
        return []
    # Already handles international domains (.in, .uk, .co.in etc.)
    email_pattern = r'\b[A-Za-z0-9._%+-]+@[A-Za-z0-9.-]+\.[A-Z|a-z]{2,10}\b'
    emails = re.findall(email_pattern, text)
    emails = [email.lower().strip() for email in emails]
    return list(set(emails))

### **Helper Function 5 : Phone Number Extraction**

Identifies and extracts phone numbers from webpage content using pattern matching. Extracted numbers are validated and standardized before being included in the final business profile.

In [10]:
def extract_phone_numbers(text):
  if not text:
    return[]
  phone_pattern = r'''
        (?:
            \+?[\d\s\-\(\)]{7,20}   # international format with optional +
            (?:ext|x|ext\.)\s*\d+   # optional extension
        |
            \+?[\d\s\-\(\)]{7,20}   # standard international
        )
    '''

  phones = re.findall(phone_pattern, text, flags=re.VERBOSE)
  cleaned = []
  for phone in phones:
    digits = re.sub(r'\D', '', phone)
    if 7 <= len(digits) <=15:
      cleaned.append(phone.strip())
  return list(set(cleaned))

In [11]:
def clean_phone_numbers(phones):
    valid_phones = []

    for phone in phones:
        digits = re.sub(r"\D", "", phone)

        if 7 <= len(digits) <= 15:
            valid_phones.append(phone.strip())
    return list(set(valid_phones))

    return valid_phones

### **Helper Function 6 : Address Extraction**

Detects and extracts physical addresses from webpage text using multiple regex patterns. Helps enrich business profiles with location information.

In [12]:
def extract_addresses(text):
    if not text:
        return []

    # Pattern 1: US addresses (existing)
    us_pattern = (
        r'\d+\s+[A-Za-z0-9\s,.-]+'
        r'(?:Ave|Avenue|St|Street|Rd|Road|Blvd|Boulevard|Dr|Drive|Ln|Lane|Way|Ct|Court)'
        r'[A-Za-z0-9\s,.-]*\d{5}'
    )

    # Pattern 2: UK addresses (postcode like SW1A 1AA)
    uk_pattern = (
        r'\d+\s+[A-Za-z0-9\s,.-]+'
        r'[A-Z]{1,2}\d{1,2}[A-Z]?\s*\d[A-Z]{2}'
    )

    # Pattern 3: Indian addresses (PIN code like 600001)
    india_pattern = (
        r'\d+[,\s]+[A-Za-z0-9\s,.-]+'
        r'(?:Street|Road|Nagar|Colony|Layout|Main|Cross|Avenue)'
        r'[A-Za-z0-9\s,.-]*\d{6}'
    )

    # Pattern 4: Generic — any line with a number and city-like words
    generic_pattern = (
        r'\d+[,\s]+[A-Za-z\s]{5,50}[,\s]+'
        r'[A-Za-z\s]{3,30}[,\s]+\d{4,6}'
    )

    addresses = []
    for pattern in [us_pattern, uk_pattern, india_pattern, generic_pattern]:
        found = re.findall(pattern, text, flags=re.IGNORECASE)
        addresses.extend([a.strip() for a in found])

    return list(set(addresses))

### **Helper Function 7 : Extract Business from Directory**

Visits HealthGrades, Yelp, and Topnpi directory pages and extracts individual business or doctor names directly from the page content. Only processes valid business listing pages - city homepages and navigation pages are automatically skipped to avoid junk results.

In [13]:
def extract_businesses_from_directory_page(url, source_name):
    """
    Visits a Yelp or Healthgrades list page and
    extracts individual business names from it.
    """
    page_text = collect_page_text(url)
    if not page_text:
        return []

    businesses = []

    if "yelp.com" in url:
        valid_yelp_patters = ["/biz", "/search?", "find_desc="]
        if not any (pattern in url for pattern in valid_yelp_patterns):
          return []

        pattern = r'\b([A-Z][a-z]+(?: [A-Z][a-z]+){1,5})\b'
        candidates = re.findall(pattern, page_text)
        skip_words = [
            "Read More", "Get Directions", "Write Review",
            "Phone Number", "Business Hours", "United States",
            "More Info", "Log In", "Sign Up", "Privacy Policy", "Terms of Service", "Cookie Policy", "Elite Squad"
        ]
        for name in candidates:
            if name not in skip and len(name) > 5:
                businesses.append({
                    "business_name": name,
                    "domain": "yelp.com",
                    "website": url,
                    "record_source": source_name
                })

    elif "healthgrades.com" in url:
        pattern = r'\b(Dr\.?\s[A-Z][a-z]+(?:\s[A-Z][a-z]+){1,3}|[A-Z][a-z]+(?: [A-Z][a-z]+){1,4}(?:Clinic|Center|Hospital|Health|Medical|Neurology|Cardiology|Associates|Group))\b'
        candidates = re.findall(pattern, page_text)
        for name in candidates:
            businesses.append({
                "business_name": name,
                "domain": "healthgrades.com",
                "website": url,
                "record_source": source_name
            })

    elif "topnpi.com" in url:
        pattern = r'(?:Dr\.\s)?([A-Z][a-z]+(?:\s[A-Z][a-z]+){1,3}),\s(?:MD|DM|DO|PHD|NP|PA)'
        candidates = re.findall(pattern, page_text)
        for name in candidates:
          businesses.append({
              "business_name": "Dr." + name,
              "domain": "topnpi.com",
              "website": url,
              "record_source": source_name
          })
    return businesses

## **Query Agent**

Defining a function which takes user's input query and spilts it into two parts - the business type and the location


In [14]:
def understand_query(user_query):
  pattern = r"\s(?:in|near to|near|around|close to)\s+"
  parts = re.split(pattern, user_query, flags=re.IGNORECASE)

  if len(parts)>=2:
    return {
      "business_type": parts[0].strip(),
      "location": parts[1].strip()
    }
  else:
    return {
      "business_type": user_query.strip(),
      "location": ""
    }

In [16]:
# Test all variations
print(understand_query("Dentists in Austin"))
print(understand_query("Dentists near Austin"))
print(understand_query("Dentists near to Austin"))
print(understand_query("Dentists around Austin"))
print(understand_query("Dentists close to Austin"))

{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}
{'business_type': 'Dentists', 'location': 'Austin'}


## **Search Query Generator**

Generates multiple search queries to gather business information from diverse online source.

In [17]:
def generate_search_queries(business_type, location):
  return[
      f"{business_type} in {location}",
      f"{business_type} in {location} official website",
      f"{business_type} in {location} reviews",
      f"{business_type} in {location} linkedin",
      f"{business_type} in {location} directory",
      f"{business_type} {location} yelp",
      f"{business_type} {location} healthgrades",
      f"{business_type} {location} site:topnpi.com"
  ]

In [18]:
result = understand_query("Neurologists in Minnesota")
queries = generate_search_queries(result["business_type"], result["location"])
for query in queries:
  print(query)

Neurologists in Minnesota
Neurologists in Minnesota official website
Neurologists in Minnesota reviews
Neurologists in Minnesota linkedin
Neurologists in Minnesota directory
Neurologists Minnesota yelp
Neurologists Minnesota healthgrades
Neurologists Minnesota site:topnpi.com


## **Search Agent**

Performs web searches using the generated queries and gathers relevant business information, including the titles, URLs and descriptions, into a structured dataframe for subsequent processing.

In [19]:
from ddgs import DDGS

def search_business(queries, max_results=5):
  all_results = []
  with DDGS() as ddgs:
    for query in queries:
      try:
        results = ddgs.text(query, max_results=max_results)
        for result in results:
          all_results.append({
            "query": query,
            "title": result.get("title", ""),
            "href": result.get("href", ""),
            "body":result.get("body", "")

        })
      except Exception as e:
        print(f"Search error:{e}")
  return pd.DataFrame(all_results)

In [20]:
result = understand_query("Neurologists in Minnesota")
queries = generate_search_queries(result["business_type"],result["location"])
all_results_df = search_business(queries, max_results=5)
all_results_df.head()

,query,title,href,body
0,Neurologists in Minnesota,The Best Neurologists in Minnesota - Health US...,https://health.usnews.com/doctors/neurologists...,We found 696 neurologists in Minnesota. The av...
1,Neurologists in Minnesota,Minnesota Monthly Recognizes UMN Neurologists ...,https://med.umn.edu/neurology/news/minnesota-m...,Minnesota Monthly Recognizes UMN Neurologists ...
2,Neurologists in Minnesota,"Neurologists & Neurosurgeons in Maple Grove, MN",https://northmemorial.com/doctor-search/neurol...,"Neurologists & Neurosurgeons in Maple Grove, M..."
3,Neurologists in Minnesota,Neurology - 11 results - Minnesota.gov,https://mn.gov/adresources/search/?query=LV-55...,Noran Neurology - Noran Neurology - Lake Elmo ...
4,Neurologists in Minnesota,Our neurologists | HealthPartners & Park Nicollet,https://www.healthpartners.com/care/find/docto...,Showing doctors and clinicians 1-20 of 79 · El...


## **Source Classfication Agent**

Analyzes the domain of each search result and categorizes it as an Official Website, Directory, Government Source, Search Engine, or Social Media platform. This classification helps assess source reliability and prioritize authoritative sources for business intelligence and profile generation.

In [21]:
def classify_source(domain):
  domain = domain.lower()
  if any(
      site in domain
      for site in [
          "linkedin",
          "facebook",
          "youtube",
          "instagram",
          "tiktok",
          "x.com",
          "twitter"
      ]
  ):
   return "Social Media"

  if any(
      site in domain
      for site in [
          "google",
          "bing",
          "duckduckgo"
      ]
  ):
    return "Search Engine"

  if any(
      site in domain
      for site in [
          "healthgrades",
          "usnews",
          "webmd",
          "medifind",
          "vitals",
          "ratemds",
          "yelp",
          "yellowpages",
          "castleconnolly",
          "threebestrated"
      ]
 ):
   return "Directory"

  if ".gov" in domain:
    return "Government"

  return "Official Website"

In [ ]:
test_domains = [
    "health.usnews.com",
    "medifind.com",
    "healthpartners.com",
    "google.com",
    "facebook.com",
    "linkedin.com"
]
for domain in test_domains:
  print(f"{domain} -> {classify_source(domain)}")

health.usnews.com -> Directory
medifind.com -> Directory
healthpartners.com -> Official Website
google.com -> Search Engine
facebook.com -> Social Media
linkedin.com -> Social Media


## **Research Sources Agent**

Enriches search results by extracting website domains and applying source classification. The agent transforms raw search results into a structured research dataset containing source metadata, enabling reliable source evaluation and downstream business profile generation.

In [22]:
def build_research_sources_df(all_results_df):
  research_sources_df = all_results_df.copy()
  research_sources_df["domain"] = (research_sources_df["href"].apply(extract_domain))
  research_sources_df["source_type"] = (research_sources_df["domain"].apply(classify_source))
  return research_sources_df

In [23]:
research_sources_df = build_research_sources_df(all_results_df)
research_sources_df[
    ["title",
    "domain",
    "source_type",
    ]
].head(10)

,title,domain,source_type
0,The Best Neurologists in Minnesota - Health US...,health.usnews.com,Directory
1,Minnesota Monthly Recognizes UMN Neurologists ...,med.umn.edu,Official Website
2,"Neurologists & Neurosurgeons in Maple Grove, MN",northmemorial.com,Official Website
3,Neurology - 11 results - Minnesota.gov,mn.gov,Government
4,Our neurologists | HealthPartners & Park Nicollet,healthpartners.com,Official Website
5,The Best Neurologists in Minnesota | US News,health.usnews.com,Directory
6,"Best Neurologists in Minneapolis, MN (2026) | ...",doctor.webmd.com,Directory
7,Best Neurologists in Minnesota (2026) | Top-Ra...,doctor.webmd.com,Directory
8,Our Neurology Providers | Minneapolis Clinic o...,minneapolisclinic.com,Official Website
9,Home :: Noran Neurology,noranclinic.com,Official Website


## **Business Discovery Agent**

Identifies real businesses from the research sources using two approches. Official Website sources use the domain name to generate a business name. Directory sources use the page title to extract the business name. Both are combined, irrelevant entries are filtered out, and individual businesses are scraped directly through HealthGrades, and Yelp directory pages to maximise the number of businesses found

In [24]:
from types import NoneType
def build_business_records_df(research_sources_df):

  exclude_keywords = ["veterinary", "directory", "directories","welli", "medicalnewstoday", "mentaltherapy", "news",
                      "blog", "americatop10", "danielaragon", "find a", "search", "best neurologists", "top neurologists",
                      "how to", "topnpi", "princeton", "scribd"]

  ui_words = ["privacy", "policy", "login", "sign up", "terms", "service", "loading", "categories", "restaurants", "copyright",
              "careers", "investors", "advertise", "support", "mobile", "developers"
              ]

  official_df = research_sources_df[research_sources_df["source_type"]=="Official Website"].copy()
  official_df["business_name"] = official_df["domain"].apply(domain_to_business_name)
  official_df["record_source"] = "Official Website"

  directory_df = research_sources_df[research_sources_df["source_type"]=="Directory"].copy()

  def extract_name_from_title(title):
    skip_phrases = [
    "best 10", "best 15", "best 20", "top 10", "top 65",
    "top neurologist", "top doctors", "find a", "search results",
    "near me", "neurologists near", "doctors near",
    "in minneapolis", "in birmingham", "in dallas",
    "in austin", "in chicago", "in houston",
    "doctors who", "physicians who", "3 best", "5 best", "10 best", "15 best", "20 best",
    "best electricians", "best plumbers", "best lawyers", "top electricians", "top plumbers"
]
    title_lower = title.lower()
    if any(phrase in title_lower for phrase in skip_phrases):
      return None
    for sep in ["|", "-", "–", "—", "·"]:
      if sep in title:
        name = title.split(sep)[0].strip()
        if len(name) > 3:
          return name
    return title.strip()

  directory_df["business_name"] = directory_df["title"].apply(extract_name_from_title)
  directory_df["record_source"] = "Directory"

  combined_df = pd.concat([official_df, directory_df], ignore_index=True)

  combined_df = combined_df[~combined_df["business_name"].str.lower().str.contains("|".join(exclude_keywords), na=False)]

  combined_df = combined_df[~combined_df["business_name"].str.lower().str.contains("|".join(ui_words), na=False)]

  combined_df = combined_df[["business_name", "domain", "href", "record_source"]].rename(columns={"href": "website"})

  combined_df = combined_df[combined_df["business_name"].str.len() > 3]
  combined_df = combined_df.reset_index(drop=True)

  combined_df = combined_df[combined_df["business_name"].notna()]
  combined_df = combined_df[combined_df["business_name"].str.len() > 3]
  combined_df = combined_df.reset_index(drop=True)

  directory_urls = research_sources_df[research_sources_df["domain"].str.contains("yelp|healthgrades|topnpi", na=False)]["href"].tolist()

  extra_businesses = []
  for url in directory_urls[:3]:
    if "yelp.com" in url:
      extra_businesses.extend(extract_businesses_from_directory_page(url, "Directory"))
    elif "healthgrades.com" in url:
      extra_businesses.extend(extract_businesses_from_directory_page(url, "Directory"))

  if extra_businesses:
        extra_df = pd.DataFrame(extra_businesses)
        combined_df = pd.concat([combined_df, extra_df], ignore_index=True)
        combined_df = combined_df.reset_index(drop=True)

  return combined_df

In [25]:
business_records_df = build_business_records_df(research_sources_df)
print("Total Businesses found:", len(business_records_df))
print("\nBy source type:")
print(business_records_df["record_source"].value_counts())
business_records_df[["business_name", "domain", "record_source"]].head(15)

Total Businesses found: 48

By source type:
record_source
Directory           44
Official Website     4
Name: count, dtype: int64


,business_name,domain,record_source
0,Northmemorial,northmemorial.com,Official Website
1,Healthpartners,healthpartners.com,Official Website
2,Minneapolisclinic,minneapolisclinic.com,Official Website
3,Noranclinic,noranclinic.com,Official Website
4,NORAN NEUROLOGY,m.yelp.com,Directory
5,NORAN NEUROLOGY,yelp.com,Directory
6,Dr. Micah Yost,healthgrades.com,Directory
7,Dr. Ilo Leppik,healthgrades.com,Directory
8,Dr. Mithri Junna,healthgrades.com,Directory
9,Dr. Steven Sabers,healthgrades.com,Directory


## **Business Deduplication Agent**

Uses RapidFuzz similarity matching to identify duplicate and remove duplicate business records. Returns the cleaned list along with the count of how many duplicates were removed.

In [26]:
from rapidfuzz import fuzz
def deduplicate_business_df(business_records_df, similarity_threshold=90):
  deduplicated=[]
  duplicates_removed = 0

  for _, row in business_records_df.iterrows():
    business_name = row["business_name"]
    is_duplicate = False

    for existing in deduplicated:
      score = fuzz.token_sort_ratio(business_name.lower(), existing["business_name"].lower())
      if score >= similarity_threshold:
        is_duplicate = True
        duplicates_removed += 1
        break

    if not is_duplicate:
      deduplicated.append(row.to_dict())

  result_df = pd.DataFrame(deduplicated, columns=business_records_df.columns)
  return result_df, duplicates_removed

In [27]:
deduplicated_business_df, duplicates_removed = deduplicate_business_df(business_records_df)
print("Before deduplication:", len(business_records_df))
print("After deduplication:", len(deduplicated_business_df))

deduplicated_business_df

Before deduplication: 48
After deduplication: 30


,business_name,domain,website,record_source
0,Northmemorial,northmemorial.com,https://northmemorial.com/doctor-search/neurol...,Official Website
1,Healthpartners,healthpartners.com,https://www.healthpartners.com/care/find/docto...,Official Website
2,Minneapolisclinic,minneapolisclinic.com,https://minneapolisclinic.com/providers/,Official Website
3,Noranclinic,noranclinic.com,https://www.noranclinic.com/,Official Website
4,NORAN NEUROLOGY,m.yelp.com,https://m.yelp.com/biz/noran-neurology-plymouth,Directory
5,Dr. Micah Yost,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
6,Dr. Ilo Leppik,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
7,Dr. Mithri Junna,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
8,Dr. Steven Sabers,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory
9,Dr. Joshua Kramer,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory


## **Business Verfication Agent**

Assigns a confidence score to each unique business based on how many independent sources it appeared in and what type of source it came from.

*  Official Website = High (>=3)
*  Directory-sourced = Medium (>=2)
*  Single-Source = Low

In [28]:
def build_verified_business_df(deduplicated_business_df, business_records_df):
  source_counts = (business_records_df.groupby("business_name")
  .size()
  .reset_index(name="source_count")
  )

  verification_df = deduplicated_business_df.merge(source_counts, on="business_name", how="left")

  def assign_confidence(row):
    if row["source_count"] >= 3 or row["record_source"] == "Official Website":
      return "High"
    elif row["source_count"] >= 2 or row["record_source"] == "Directory":
      return "Medium"
    else:
      return "Low"

  verification_df["confidence"] = verification_df.apply(assign_confidence, axis=1)
  return verification_df

In [29]:
verification_df = build_verified_business_df(deduplicated_business_df, business_records_df)
print("Total verified businesses:", len(verification_df))
print("\nConfidence breakdown:")
print(verification_df["confidence"].value_counts())
verification_df.head(10)

Total verified businesses: 30

Confidence breakdown:
confidence
Medium    24
High       6
Name: count, dtype: int64


,business_name,domain,website,record_source,source_count,confidence
0,Northmemorial,northmemorial.com,https://northmemorial.com/doctor-search/neurol...,Official Website,1,High
1,Healthpartners,healthpartners.com,https://www.healthpartners.com/care/find/docto...,Official Website,1,High
2,Minneapolisclinic,minneapolisclinic.com,https://minneapolisclinic.com/providers/,Official Website,1,High
3,Noranclinic,noranclinic.com,https://www.noranclinic.com/,Official Website,1,High
4,NORAN NEUROLOGY,m.yelp.com,https://m.yelp.com/biz/noran-neurology-plymouth,Directory,2,Medium
5,Dr. Micah Yost,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
6,Dr. Ilo Leppik,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,2,Medium
7,Dr. Mithri Junna,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
8,Dr. Steven Sabers,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,1,Medium
9,Dr. Joshua Kramer,healthgrades.com,https://www.healthgrades.com/neurology-directo...,Directory,3,High


## **Contact Information Agent**

Visits verified business websites and extracts contact information including addresses, phone numbers, and email addresses. The agent enriches business records with publicly available contact details to support business outreach and profile generation.

In [30]:
def build_contact_df(verification_df):
  contact_records = []
  page_cache = {}
  for idx, row in verification_df.iterrows():
    try:
      website = row["website"]
      if website in page_cache:
        page_text = page_cache[website]
      else:
        page_text = collect_page_text(website)
      page_cache[website] = page_text

      if page_text:
        phones = extract_phone_numbers(page_text)
        phones = clean_phone_numbers(phones)
        emails = extract_emails(page_text)
        addresses = extract_addresses(page_text)
      else:
        phones = []
        emails = []
        addresses = []

      contact_records.append({
          "business_name": row["business_name"],
          "address": addresses[0] if addresses else None,
          "phones": phones,
          "emails": emails,
          "source_url": website
      })
      print(f"{idx}: {row['business_name']}")

    except Exception as e:
      print(f"{idx}: {row['business_name']}, - ERROR: {e}")
      contact_records.append({
          "business_name": row["business_name"],
          "address": None,
          "phones": [],
          "emails": [],
          "source_url": row.get("website", "")
      })
  return pd.DataFrame(contact_records)

In [31]:
contact_df = build_contact_df(verification_df)
print("Business processed:", len(contact_df))
print("Business with phones:", contact_df["phones"].apply(len).gt(0).sum())
print("Business with emails:", contact_df["emails"].apply(len).gt(0).sum())
print("Business with addresses:", contact_df["address"].notna().sum())
contact_df.head(10)

0: Northmemorial
1: Healthpartners
2: Minneapolisclinic
3: Noranclinic
4: NORAN NEUROLOGY
5: Dr. Micah Yost
6: Dr. Ilo Leppik
7: Dr. Mithri Junna
8: Dr. Steven Sabers
9: Dr. Joshua Kramer
10: Dr. Kenneth Hoj
11: Dr. Dimitrios Giannakidis
12: Dr. Carrie Robertson
13: Dr. Yumna Saeed
14: Dr. Fred Lux
15: Dr. Syed Shahkhan
16: Dr. Rupert Exconde
17: Dr. Nadeem Iqbal
18: Dr. Oliver Ni
19: Dr. Rwoof Reshi
20: Dr. Sotirios Parashos
21: Dr. Ryan Coburn
22: Dr. Eleanor Orehek
23: Dr. Thomas Schriefer
24: Dr. Fred Cutrer
25: Dr. Kevin Webb
26: Dr Golden Valley
27: Dr. Saugat Dey
28: Dr. Rammohan Sankaraneni
29: Dr. William Schmalstieg
Business processed: 30
Business with phones: 27
Business with emails: 1
Business with addresses: 27


,business_name,address,phones,emails,source_url
0,Northmemorial,"11 Maple Grove, MN 55369","[612-589-4287, 55330 763-581-5200, 55427 763-5...",[careers@northmemorial.com],https://northmemorial.com/doctor-search/neurol...
1,Healthpartners,"8170 33rd Ave S, Bloomington, MN 55425",[],[],https://www.healthpartners.com/care/find/docto...
2,Minneapolisclinic,None,[],[],https://minneapolisclinic.com/providers/
3,Noranclinic,None,"[(612) 879-1000, (612) 879-0722]",[],https://www.noranclinic.com/
4,NORAN NEUROLOGY,None,[],[],https://m.yelp.com/biz/noran-neurology-plymouth
5,Dr. Micah Yost,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...
6,Dr. Ilo Leppik,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...
7,Dr. Mithri Junna,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...
8,Dr. Steven Sabers,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...
9,Dr. Joshua Kramer,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...


## **Final Output Agent**

Combines verified business records and extracted contact information into a unified business intelligence dataset. The agent consolidates business details, contact information, source evidence, and confidence scores into a structured final output for analysis and decision-making.

In [32]:
def build_final_output_df(verification_df, contact_df):
  final_output_df = (verification_df.merge(contact_df, on="business_name", how="left"))
  return final_output_df[
      [
          "business_name",
          "address",
          "phones",
          "emails",
          "website",
          "source_count",
          "confidence",
          "record_source",
          "source_url"
      ]
  ]

In [33]:
final_output_df = build_final_output_df(verification_df, contact_df)
print("Total Businesses:", len(final_output_df))
print("High Confidence Businesses:", (final_output_df["confidence"]=="High").sum())
print("Medium Confidence Businesses:", (final_output_df["confidence"]=="Medium").sum())
print("\nColumns:", final_output_df.columns.tolist())
final_output_df

Total Businesses: 30
High Confidence Businesses: 6
Medium Confidence Businesses: 24

Columns: ['business_name', 'address', 'phones', 'emails', 'website', 'source_count', 'confidence', 'record_source', 'source_url']


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Northmemorial,"11 Maple Grove, MN 55369","[612-589-4287, 55330 763-581-5200, 55427 763-5...",[careers@northmemorial.com],https://northmemorial.com/doctor-search/neurol...,1,High,Official Website,https://northmemorial.com/doctor-search/neurol...
1,Healthpartners,"8170 33rd Ave S, Bloomington, MN 55425",[],[],https://www.healthpartners.com/care/find/docto...,1,High,Official Website,https://www.healthpartners.com/care/find/docto...
2,Minneapolisclinic,None,[],[],https://minneapolisclinic.com/providers/,1,High,Official Website,https://minneapolisclinic.com/providers/
3,Noranclinic,None,"[(612) 879-1000, (612) 879-0722]",[],https://www.noranclinic.com/,1,High,Official Website,https://www.noranclinic.com/
4,NORAN NEUROLOGY,None,[],[],https://m.yelp.com/biz/noran-neurology-plymouth,2,Medium,Directory,https://m.yelp.com/biz/noran-neurology-plymouth
5,Dr. Micah Yost,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
6,Dr. Ilo Leppik,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,2,Medium,Directory,https://www.healthgrades.com/neurology-directo...
7,Dr. Mithri Junna,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
8,Dr. Steven Sabers,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
9,Dr. Joshua Kramer,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,3,High,Directory,https://www.healthgrades.com/neurology-directo...


## **Research Summary Agent**

Sends the final business dataset to Gemini with a structured prompt, which generates a professional research report in Markdown format. The report includes a Search Summary, Executive Summary, Key Businesses Identified, Research Source Overview, Data Quality Summary, Important Findings, and Conclusion.

In [34]:
def generate_research_summary(user_query, final_output_df, research_sources_df, duplicates_removed=0):
  business_details = final_output_df.to_dict("records")
  source_types = (research_sources_df["source_type"]
                  .value_counts()
                  .to_dict()
                  )
  total = len(final_output_df)
  with_phone = final_output_df["phones"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()
  with_address = final_output_df["address"].notna().sum()
  with_email = final_output_df["emails"].apply(lambda x: len(x) > 0 if isinstance(x, list) else False).sum()

  prompt = f""" You are a senior business research analyst.

  User Query:{user_query}

  Search statistics:
  -- Businesses Found : {total + duplicates_removed}
  -- Businesses Verified : {total}
  -- Duplicate Records Removed: {duplicates_removed}
  -- Sources Searched : {len(research_sources_df)}

  Data Quality:
  -- Records with Phone : {round(with_phone/total*100)}%
  -- Records with Address : {round(with_address/total*100)}%
  -- Records with Email : {round(with_email/total*100)}%

  Business Details: {business_details}

  Research Source Distribution: {source_types}

  Generate a professional business research report in Markdown format.

  Use Exactly the following structure:

  # Business Research Report

  ## Search Summary
  -- Query : {user_query}
  -- Businesses Found : {total + duplicates_removed}
  -- Businesses Verified : {total}
  -- Duplicate Records Removed: {duplicates_removed}
  -- Sources Searched : {len(research_sources_df)}

  ## Executive Summary
  provide a concise overview of the market and research findings.

  ## Key Business Identified
  For each business include:
  -- Business Name
  -- Website
  -- Source Count
  -- Confidence Level
  -- Address (if available)
  -- Phone Numbers (if available)

Important:
-- Report Source Count exactly as provided in the dataset.
-- Report Confidence Level exactly as provided in the dataset.
-- Do not modify, reinterpret, estimate, or recalculate confidence values.
-- Do NOT generate business descriptions.
-- Only report information explicitly present in the provided dataset.
-- If a field is unavailable, write "Not Available",


  ## Research Source Overview
  Include:
  -- Total sources analyzed
  -- Breakdown of source types
  -- observations about source reliability

  ## Data Quality Summary
  -- Records with Phone Number : {round(with_phone/total*100)}%
  -- Records with Address : {round(with_address/total*100)}%
  -- Records with Email : {round(with_email/total*100)}%

## Important Findings
Provide 3-5 bullet point insights.

## Conclusion
Provide a brief concluding paragraph.

Formatting Requirements:
-- Use proper Markdown headings(#, ##).
-- Use bullet points where appropriate.
-- Keep the report professional and concise.
-- Do not invent facts that are not supported by the provided data.


  """

  response = model.generate_content(prompt)
  return response.text

In [35]:
research_summary = generate_research_summary(
    user_query="Neurologists in Minnesota",
    final_output_df=final_output_df,
    research_sources_df=research_sources_df,
    duplicates_removed=duplicates_removed
)
from IPython.display import Markdown, display
display(Markdown(research_summary))

# Business Research Report

## Search Summary
-- Query : Neurologists in Minnesota
-- Businesses Found : 48
-- Businesses Verified : 30
-- Duplicate Records Removed: 18
-- Sources Searched : 40

## Executive Summary
This report presents the findings from a business research query for "Neurologists in Minnesota." A total of 48 businesses were initially identified, with 30 successfully verified after removing 18 duplicate records. The research utilized 40 different sources. While phone numbers and addresses are highly available for verified records (90% each), email addresses are scarce, present in only 3% of records. Key businesses identified include large healthcare systems and individual neurologists, with varying confidence levels based on the number and type of sources.

## Key Business Identified
-- Business Name: Northmemorial
-- Website: https://northmemorial.com/doctor-search/neurologists-neurosurgeons-maple-grove-mn/
-- Source Count: 1
-- Confidence Level: High
-- Address: 11 Maple Grove, MN 55369
-- Phone Numbers: 612-589-4287, 55330 763-581-5200, 55427 763-581-5150, 55427 763-581-5700, 55421 763-581-5500, 55449 763-581-5951, 55369 763-581-5050, 763-581-1000, 4181 108, 55369 763-581-2800, 55345 763-581-5400, 55422 763-581-5400, 55345 763-581-8900, 763-581-1025, 55369 763-581-2035, 55443 763-581-5660, 55362 763-271-2800, 55369 763-581-9220, 55369 763-581-1000, 55422 763-520-5200, 763-520-5200, 55362 763-581-5400, 55412 763-581-5750, 55449 763-581-5400, 55369 763-581-5400, 55369 763-581-5800, 763-581-4654, 13800 83, 55422 763-581-2800, 763-581-0780, 55430 763-581-5630, 55369 763-581-5900, 55422 763-581-3550, 55432 763-581-0600, 55449 763-581-5952

-- Business Name: Healthpartners
-- Website: https://www.healthpartners.com/care/find/doctors/neurologists/
-- Source Count: 1
-- Confidence Level: High
-- Address: 8170 33rd Ave S, Bloomington, MN 55425
-- Phone Numbers: Not Available

-- Business Name: Minneapolisclinic
-- Website: https://minneapolisclinic.com/providers/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Noranclinic
-- Website: https://www.noranclinic.com/
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: (612) 879-1000, (612) 879-0722

-- Business Name: NORAN NEUROLOGY
-- Website: https://m.yelp.com/biz/noran-neurology-plymouth
-- Source Count: 2
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Micah Yost
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Ilo Leppik
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Mithri Junna
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Steven Sabers
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Joshua Kramer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 3
-- Confidence Level: High
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Kenneth Hoj
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Dimitrios Giannakidis
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Carrie Robertson
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Yumna Saeed
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Fred Lux
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Syed Shahkhan
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Rupert Exconde
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Nadeem Iqbal
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Oliver Ni
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Rwoof Reshi
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Sotirios Parashos
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Ryan Coburn
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Eleanor Orehek
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Thomas Schriefer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 2
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Fred Cutrer
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- Phone Numbers: (612) 473-0657, 55812 (218) 206-634

-- Business Name: Dr. Kevin Webb
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 717 Delaware St SE Rm 511 Minneapolis, MN 55414 1.8 mi miles away Offers Telehealth 717 Delaware St SE Rm 511 Minneapolis, MN 55414
-- Phone Numbers: (612) 482-0661

-- Business Name: Dr Golden Valley
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 4
-- Confidence Level: High
-- Address: 717 Delaware St SE Rm 511 Minneapolis, MN 55414 1.8 mi miles away Offers Telehealth 717 Delaware St SE Rm 511 Minneapolis, MN 55414
-- Phone Numbers: (612) 482-0661

-- Business Name: Dr. Saugat Dey
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 717 Delaware St SE Rm 511 Minneapolis, MN 55414 1.8 mi miles away Offers Telehealth 717 Delaware St SE Rm 511 Minneapolis, MN 55414
-- Phone Numbers: (612) 482-0661

-- Business Name: Dr. Rammohan Sankaraneni
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 717 Delaware St SE Rm 511 Minneapolis, MN 55414 1.8 mi miles away Offers Telehealth 717 Delaware St SE Rm 511 Minneapolis, MN 55414
-- Phone Numbers: (612) 482-0661

-- Business Name: Dr. William Schmalstieg
-- Website: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- Source Count: 1
-- Confidence Level: Medium
-- Address: 717 Delaware St SE Rm 511 Minneapolis, MN 55414 1.8 mi miles away Offers Telehealth 717 Delaware St SE Rm 511 Minneapolis, MN 55414
-- Phone Numbers: (612) 482-0661

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    *   Directory: 27
    *   Official Website: 11
    *   Government: 1
    *   Social Media: 1
-- observations about source reliability: The majority of identified records (27 out of 40 sources) originated from directories, which generally provide medium confidence data. Official websites, offering high confidence, contributed 11 sources. Government and Social Media sources were minimal. This distribution suggests a need for careful verification of directory-sourced information due to its typically lower reliability compared to official channels.

## Data Quality Summary
-- Records with Phone Number : 90%
-- Records with Address : 90%
-- Records with Email : 3%

## Important Findings
*   A significant number of initial business records (18 out of 48) were duplicates, indicating potential redundancy across various data sources.
*   While contact information like phone numbers and addresses are highly available (90%), email addresses are extremely scarce (3%), which could be a challenge for direct digital outreach.
*   The majority of verified records are individual neurologists listed under a single primary address on Healthgrades.com, suggesting a concentration of practitioners in certain locations or reliance on specific directories for data aggregation.
*   Large healthcare organizations like Northmemorial and Healthpartners have a presence, confirmed by their official websites, offering high confidence data.
*   The high proportion of "Directory" sources (27 out of 40) for verification implies that much of the data relies on third-party listings, which may require cross-referencing for optimal accuracy.

## Conclusion
The research successfully identified 30 verified neurologists or neurological practices in Minnesota from an initial pool of 48, using 40 distinct sources. While contact information such as phone numbers and addresses is largely available and reliable, the notable absence of email addresses presents a limitation for certain outreach strategies. The reliance on directory sources suggests that ongoing data hygiene and direct verification efforts would be beneficial to maintain high accuracy.

## **Master Agent**

Orchestrates the entire search pipeline with a single function call. Takes any user query, runs it through all agents in sequence - from query understanding to report generation - and returns all intermediate and final outputs. Supports any business type and location, including international queries.

In [43]:
def run_agent(user_query):
  query_info = understand_query(user_query)
  business_type = query_info["business_type"]
  location = query_info["location"]

  queries = generate_search_queries(business_type, location)
  all_results_df = search_business(queries, max_results=5)
  research_sources_df = build_research_sources_df(all_results_df)
  business_records_df = build_business_records_df(research_sources_df)

  deduplicated_business_df, duplicates_removed = deduplicate_business_df(business_records_df)

  verification_df = build_verified_business_df(deduplicated_business_df, business_records_df)
  contact_df = build_contact_df(verification_df)
  final_output_df = build_final_output_df(verification_df, contact_df)

  research_summary = generate_research_summary(
      user_query=user_query,
      final_output_df=final_output_df,
      research_sources_df=research_sources_df,
      duplicates_removed = duplicates_removed
  )

  return {
      "research_sources_df": research_sources_df,
      "business_records_df": business_records_df,
      "deduplicated_business_df": deduplicated_business_df,
      "duplicates_removed": duplicates_removed,
      "verification_df": verification_df,
      "contact_df": contact_df,
      "final_output_df": final_output_df,
      "research_summary": research_summary
  }

In [44]:
results = run_agent("Neurologists in Minnesota")
results["final_output_df"]

0: Healthpartners
1: NORAN NEUROLOGY
2: Dr. Micah Yost
3: Dr. Ilo Leppik
4: Dr. Mithri Junna
5: Dr. Steven Sabers
6: Dr. Joshua Kramer
7: Dr. Kenneth Hoj
8: Dr. Dimitrios Giannakidis
9: Dr. Carrie Robertson
10: Dr. Yumna Saeed
11: Dr. Fred Lux
12: Dr. Syed Shahkhan
13: Dr. Rupert Exconde
14: Dr. Nadeem Iqbal
15: Dr. Oliver Ni
16: Dr. Rwoof Reshi
17: Dr. Sotirios Parashos
18: Dr. Ryan Coburn
19: Dr. Eleanor Orehek
20: Dr. Thomas Schriefer
21: Dr. Fred Cutrer
22: Dr Golden Valley
23: Dr. Saugat Dey
24: Dr. Rammohan Sankaraneni
25: Dr. William Schmalstieg
26: Dr. Gerald Dove
27: Dr. Susan Minette
28: Dr. John Damergis
29: Dr. Scott Bundlie


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 1865.89ms


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Healthpartners,"8170 33rd Ave S, Bloomington, MN 55425",[],[],https://www.healthpartners.com/care/find/docto...,1,High,Official Website,https://www.healthpartners.com/care/find/docto...
1,NORAN NEUROLOGY,None,[],[],https://m.yelp.com/biz/noran-neurology-plymouth,1,Medium,Directory,https://m.yelp.com/biz/noran-neurology-plymouth
2,Dr. Micah Yost,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
3,Dr. Ilo Leppik,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,2,Medium,Directory,https://www.healthgrades.com/neurology-directo...
4,Dr. Mithri Junna,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
5,Dr. Steven Sabers,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...
6,Dr. Joshua Kramer,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,3,High,Directory,https://www.healthgrades.com/neurology-directo...
7,Dr. Kenneth Hoj,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,2,Medium,Directory,https://www.healthgrades.com/neurology-directo...
8,Dr. Dimitrios Giannakidis,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,2,Medium,Directory,https://www.healthgrades.com/neurology-directo...
9,Dr. Carrie Robertson,3 more provider attributes 11091 Ulysses St NE...,"[(612) 473-0657, 55812 (218) 206-634]",[],https://www.healthgrades.com/neurology-directo...,1,Medium,Directory,https://www.healthgrades.com/neurology-directo...


In [45]:
from IPython.display import Markdown, display
display(Markdown(results["research_summary"]))

# Business Research Report

## Search Summary
-- Query : Neurologists in Minnesota
-- Businesses Found : 45
-- Businesses Verified : 30
-- Duplicate Records Removed: 15
-- Sources Searched : 40

## Executive Summary
This report provides a comprehensive overview of neurologists identified in Minnesota. The search successfully located 45 businesses or individual practitioners, with 30 records verified and 15 duplicate entries removed, indicating a robust initial data collection effort. While contact information for phone numbers (93%) and addresses (97%) is nearly complete across verified records, there is a notable absence of email addresses (0%). The identified businesses range from large healthcare systems to individual practitioners, predominantly sourced from directories and official websites, offering a solid foundation for outreach or further market analysis.

## Key Business Identified
-- **Business Name**: Healthpartners
-- **Website**: https://www.healthpartners.com/care/find/doctors/neurologists/
-- **Source Count**: 1
-- **Confidence Level**: High
-- **Address**: 8170 33rd Ave S, Bloomington, MN 55425
-- **Phone Numbers**: Not Available

-- **Business Name**: NORAN NEUROLOGY
-- **Website**: https://m.yelp.com/biz/noran-neurology-plymouth
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: Not Available
-- **Phone Numbers**: Not Available

-- **Business Name**: Dr. Micah Yost
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Ilo Leppik
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Mithri Junna
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Steven Sabers
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Joshua Kramer
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 3
-- **Confidence Level**: High
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Kenneth Hoj
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Dimitrios Giannakidis
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Carrie Robertson
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Yumna Saeed
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Fred Lux
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Syed Shahkhan
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Rupert Exconde
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Nadeem Iqbal
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Oliver Ni
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Rwoof Reshi
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Sotirios Parashos
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Ryan Coburn
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Eleanor Orehek
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Thomas Schriefer
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr. Fred Cutrer
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3 more provider attributes 11091 Ulysses St NE Ste 100 Blaine, MN 55434
-- **Phone Numbers**: (612) 473-0657, 55812 (218) 206-634

-- **Business Name**: Dr Golden Valley
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 2
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. Saugat Dey
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. Rammohan Sankaraneni
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. William Schmalstieg
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. Gerald Dove
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. Susan Minette
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. John Damergis
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

-- **Business Name**: Dr. Scott Bundlie
-- **Website**: https://www.healthgrades.com/neurology-directory/mn-minnesota/minneapolis
-- **Source Count**: 1
-- **Confidence Level**: Medium
-- **Address**: 3601 Minnesota Dr Ste 200 Minneapolis, MN 55435
-- **Phone Numbers**: (612) 473-7023, (612) 473-7008, (612) 504-9387, (612) 482-0661

## Research Source Overview
-- **Total sources analyzed**: 40
-- **Breakdown of source types**:
    -- Directory: 28
    -- Official Website: 7
    -- Social Media: 5
-- **Observations about source reliability**: The majority of information was gathered from 'Directory' sources (28 records), notably Healthgrades, which provided listings for numerous individual neurologists. 'Official Website' sources (7 records) provided high-confidence data, although less frequently. 'Social Media' (5 records) contributed a smaller portion. The prevalence of directory sources suggests a broad initial sweep, with the potential for higher confidence when cross-referenced or confirmed by official websites.

## Data Quality Summary
-- Records with Phone Number : 93%
-- Records with Address : 97%
-- Records with Email : 0%

## Important Findings
*   A significant number of neurologists in Minnesota have been identified and verified, demonstrating good coverage for the query.
*   Contact information for phone numbers and addresses is highly comprehensive, making direct outreach or geographical analysis feasible.
*   The complete absence of email addresses across all records presents a significant limitation for digital communication strategies.
*   Many individual neurologists are listed with the same address and phone numbers, particularly from Healthgrades, suggesting they practice at shared clinic locations or are part of larger group practices.
*   The search identified both large healthcare providers like Healthpartners and individual practitioners, providing a diverse list of neurological services.

## Conclusion
The research successfully compiled a substantial list of neurologists in Minnesota, characterized by a high verification rate and excellent coverage of physical addresses and phone numbers. While the lack of email addresses is a key data gap, the available contact information provides a strong foundation for various business objectives, including market mapping, direct mail campaigns, or telemarketing efforts targeting this professional group. Further investigation might be required to obtain direct email contacts for these practitioners.

In [46]:
results = run_agent("Cardiologists in Birmingham")
display(results["final_output_df"])
from IPython.display import Markdown, display
display(Markdown(results["research_summary"]))

0: Uabstvincents
1: Birminghamheart
2: Baptisthealthal
3: Doctify
4: Doctorshire
5: Top 60 Cardiologists near Birmingham, AL
6: BIRMINGHAM HEART CLINIC PC
7: STEPHEN BAKIR, MD
8: BYRON JONES, MD
9: ALABAMA CARDIOVASCULAR GROUP
10: Heart doctors and cardiologists near Birmingham, AL
11: Dr. Mustafa Ahmed
12: Dr. Monica Hunter
13: Dr. John Eagan
14: Dr. Juan Bernal
15: Dr. Barry Rayburn
16: Dr. Dale Kirby
17: Dr. William Maddox
18: Dr. Hutton Brantley
19: Dr. Edward Cullum
20: Dr. Michael Honan
21: Dr. Stephen Bakir
22: Dr. Raashid Ashraf
23: Dr. Vikram Arora
24: Dr. Mohamed Jasser
25: Dr. David Schultz
26: Dr. Benjamin Plaisance
27: Dr. Alan Gertler
28: Dr. Hasan Guven
29: Dr. Christopher King


ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 2521.46ms
ERROR:tornado.access:503 POST /v1beta/models/gemini-2.5-flash:generateContent?%24alt=json%3Benum-encoding%3Dint (::1) 3712.23ms


,business_name,address,phones,emails,website,source_count,confidence,record_source,source_url
0,Uabstvincents,"7191 Cahaba Valley Rd., Suite 106 Birmingham, ...","[35244 205-939-7100, 35205 205-939-7100, 35071...",[],https://uabstvincents.org/locations/cardiology...,2,High,Official Website,https://uabstvincents.org/locations/cardiology...
1,Birminghamheart,None,[],[],https://birminghamheart.com/,2,High,Official Website,https://birminghamheart.com/
2,Baptisthealthal,None,"[5 10 20 30 40 50 75, 100 150 200, (210) 510-5...",[],https://www.baptisthealthal.com/cva,1,High,Official Website,https://www.baptisthealthal.com/cva
3,Doctify,None,[],[],https://www.doctify.com/uk/find/cardiology/bir...,1,High,Official Website,https://www.doctify.com/uk/find/cardiology/bir...
4,Doctorshire,None,[],[],https://www.doctorshire.com/doctors/birmingham...,1,High,Official Website,https://www.doctorshire.com/doctors/birmingham...
5,"Top 60 Cardiologists near Birmingham, AL",None,[],[],https://www.vitals.com/cardiovascular-disease/...,1,Medium,Directory,https://www.vitals.com/cardiovascular-disease/...
6,BIRMINGHAM HEART CLINIC PC,None,[],[],https://m.yelp.com/biz/birmingham-heart-clinic...,1,Medium,Directory,https://m.yelp.com/biz/birmingham-heart-clinic...
7,"STEPHEN BAKIR, MD",None,[],[],https://m.yelp.com/biz/stephen-bakir-md-birmin...,1,Medium,Directory,https://m.yelp.com/biz/stephen-bakir-md-birmin...
8,"BYRON JONES, MD",None,[],[],https://m.yelp.com/biz/byron-jones-md-birmingham,1,Medium,Directory,https://m.yelp.com/biz/byron-jones-md-birmingham
9,ALABAMA CARDIOVASCULAR GROUP,None,[],[],https://m.yelp.com/biz/alabama-cardiovascular-...,1,Medium,Directory,https://m.yelp.com/biz/alabama-cardiovascular-...


# Business Research Report

## Search Summary
-- Query : Cardiologists in Birmingham
-- Businesses Found : 70
-- Businesses Verified : 30
-- Duplicate Records Removed: 40
-- Sources Searched : 40

## Executive Summary
This report presents the findings of a business research query for "Cardiologists in Birmingham." Out of 70 businesses initially found, 30 unique and verified records were identified after removing 40 duplicate entries. The research leveraged 40 different sources. While address data is reasonably present (67%), key contact information such as phone numbers is scarce (7%), and email addresses are entirely absent (0%), indicating significant data quality challenges for direct outreach or detailed contact analysis.

## Key Business Identified

-- Business Name: Uabstvincents
-- Website: https://uabstvincents.org/locations/cardiology-specialists-of-birmingham/
-- Source Count: 2
-- Confidence Level: High
-- Address: 7191 Cahaba Valley Rd., Suite 106 Birmingham, AL 35242 205-939-7100 1130 22nd Street Birmingham, AL 35205
-- Phone Numbers: 35244 205-939-7100, 35205 205-939-7100, 35071 205-939-7100, 35242 205-939-7100

-- Business Name: Birminghamheart
-- Website: https://birminghamheart.com/
-- Source Count: 2
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Baptisthealthal
-- Website: https://www.baptisthealthal.com/cva
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: 5 10 20 30 40 50 75, 100 150 200, (210) 510-5000, (205) 776-8600, (205) 510-5000

-- Business Name: Doctify
-- Website: https://www.doctify.com/uk/find/cardiology/birmingham/specialists
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Doctorshire
-- Website: https://www.doctorshire.com/doctors/birmingham/cardiologist
-- Source Count: 1
-- Confidence Level: High
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Top 60 Cardiologists near Birmingham, AL
-- Website: https://www.vitals.com/cardiovascular-disease/al/birmingham
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: BIRMINGHAM HEART CLINIC PC
-- Website: https://m.yelp.com/biz/birmingham-heart-clinic-pc-birmingham-2
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: STEPHEN BAKIR, MD
-- Website: https://m.yelp.com/biz/stephen-bakir-md-birmingham
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: BYRON JONES, MD
-- Website: https://m.yelp.com/biz/byron-jones-md-birmingham
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: ALABAMA CARDIOVASCULAR GROUP
-- Website: https://m.yelp.com/biz/alabama-cardiovascular-group-birmingham-2
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Heart doctors and cardiologists near Birmingham, AL
-- Website: https://health.usnews.com/doctors/cardiologists/alabama/birmingham
-- Source Count: 1
-- Confidence Level: Medium
-- Address: Not Available
-- Phone Numbers: Not Available

-- Business Name: Dr. Mustafa Ahmed
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Monica Hunter
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. John Eagan
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Juan Bernal
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Barry Rayburn
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Dale Kirby
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. William Maddox
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Hutton Brantley
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Edward Cullum
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Michael Honan
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Stephen Bakir
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Raashid Ashraf
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Vikram Arora
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Mohamed Jasser
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. David Schultz
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Benjamin Plaisance
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Alan Gertler
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Hasan Guven
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

-- Business Name: Dr. Christopher King
-- Website: https://www.healthgrades.com/cardiology-directory/al-alabama/birmingham
-- Source Count: 3
-- Confidence Level: High
-- Address: 2000 6th Ave S Birmingham, AL 35233
-- Phone Numbers: Not Available

## Research Source Overview
-- Total sources analyzed: 40
-- Breakdown of source types:
    *   Directory: 20
    *   Official Website: 15
    *   Social Media: 5
-- Observations about source reliability: The majority of identified businesses stem from Directory and Official Website sources, which generally exhibit higher confidence levels (High or Medium). Social Media sources represent a smaller portion of the overall search.

## Data Quality Summary
-- Records with Phone Number : 7%
-- Records with Address : 67%
-- Records with Email : 0%

## Important Findings
*   A significant amount of initial data (40 out of 70 records) consisted of duplicate entries, indicating a need for robust de-duplication processes.
*   Direct contact information is extremely limited, with only 7% of verified records including phone numbers and no records providing email addresses.
*   Address information is available for a majority (67%) of the verified businesses, suggesting geographic targeting is possible.
*   Many individual cardiologists were identified through directory sources like Healthgrades, but their individual phone and email contact details were not available in the dataset.
*   Official Websites and Directories are the primary sources of verified information, often yielding high confidence levels.

## Conclusion
The research successfully identified 30 verified cardiologists or cardiology-related entities in Birmingham, predominantly leveraging directory and official website sources. While geographical data is reasonably available, the severe lack of direct contact information (phone and especially email) presents a significant challenge for any outreach or direct communication efforts, highlighting a critical data gap.